In [1]:
import pandas as pd
data = pd.read_csv('ner_dataset.csv', encoding= 'unicode_escape')
data.head()

,Sentence #,Word,POS,Tag
0,Sentence: 1,Thousands,NNS,O
1,NaN,of,IN,O
2,NaN,demonstrators,NNS,O
3,NaN,have,VBP,O
4,NaN,marched,VBN,O


In [2]:
from itertools import chain
def get_dict_map(data, token_or_tag):
    tok2idx = {}
    idx2tok = {}
    
    if token_or_tag == 'token':
        vocab = list(set(data['Word'].to_list()))
    else:
        vocab = list(set(data['Tag'].to_list()))
    
    idx2tok = {idx:tok for  idx, tok in enumerate(vocab)}
    tok2idx = {tok:idx for  idx, tok in enumerate(vocab)}
    return tok2idx, idx2tok
token2idx, idx2token = get_dict_map(data, 'token')
tag2idx, idx2tag = get_dict_map(data, 'tag')

In [3]:
data['Word_idx'] = data['Word'].map(token2idx)
data['Tag_idx'] = data['Tag'].map(tag2idx)
data_fillna = data.ffill(axis=0)
# Groupby and collect columns
data_group = data_fillna.groupby(
['Sentence #'],as_index=False
)[['Word', 'POS', 'Tag', 'Word_idx', 'Tag_idx']].agg(lambda x: list(x))

In [4]:
from sklearn.model_selection import train_test_split
from keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.utils import to_categorical

def get_pad_train_test_val(data_group, data):

    #get max token and tag length
    n_token = len(list(set(data['Word'].to_list())))
    n_tag = len(list(set(data['Tag'].to_list())))

    #Pad tokens (X var)    
    tokens = data_group['Word_idx'].tolist()
    maxlen = max([len(s) for s in tokens])
    pad_tokens = pad_sequences(tokens, maxlen=maxlen, dtype='int32', padding='post', value= n_token - 1)

    #Pad Tags (y var) and convert it into one hot encoding
    tags = data_group['Tag_idx'].tolist()
    pad_tags = pad_sequences(tags, maxlen=maxlen, dtype='int32', padding='post', value= tag2idx["O"])
    n_tags = len(tag2idx)
    pad_tags = [to_categorical(i, num_classes=n_tags) for i in pad_tags]
    
    #Split train, test and validation set
    tokens_, test_tokens, tags_, test_tags = train_test_split(pad_tokens, pad_tags, test_size=0.1, train_size=0.9, random_state=2020)
    train_tokens, val_tokens, train_tags, val_tags = train_test_split(tokens_,tags_,test_size = 0.25,train_size =0.75, random_state=2020)

    print(
        'train_tokens length:', len(train_tokens),
        '\ntrain_tokens length:', len(train_tokens),
        '\ntest_tokens length:', len(test_tokens),
        '\ntest_tags:', len(test_tags),
        '\nval_tokens:', len(val_tokens),
        '\nval_tags:', len(val_tags),
    )
    
    return train_tokens, val_tokens, test_tokens, train_tags, val_tags, test_tags

train_tokens, val_tokens, test_tokens, train_tags, val_tags, test_tags = get_pad_train_test_val(data_group, data)

train_tokens length: 32372 
train_tokens length: 32372 
test_tokens length: 4796 
test_tags: 4796 
val_tokens: 10791 
val_tags: 10791


In [5]:
import numpy as np
import tensorflow
from tensorflow.keras import Sequential, Model, Input
from tensorflow.keras.layers import LSTM, Embedding, Dense, TimeDistributed, Dropout, Bidirectional
from tensorflow.keras.utils import plot_model
from numpy.random import seed
seed(1)
tensorflow.random.set_seed(2)

In [6]:
input_dim = len(list(set(data['Word'].to_list())))+1
output_dim = 64
input_length = max([len(s) for s in data_group['Word_idx'].tolist()])
n_tags = len(tag2idx)

In [7]:
def get_bilstm_lstm_model():
    model = Sequential()

    # Explicitly define the input shape
    model.add(Input(shape=(input_length,), dtype="int32"))

    # Embedding layer
    model.add(
        Embedding(
            input_dim=input_dim,
            output_dim=output_dim
        )
    )

    # Bidirectional LSTM
    model.add(
        Bidirectional(
            LSTM(
                units=output_dim,
                return_sequences=True,
                dropout=0.2,
                recurrent_dropout=0.2
            ),
            merge_mode="concat"
        )
    )

    # Additional LSTM layer
    model.add(
        LSTM(
            units=output_dim,
            return_sequences=True,
            dropout=0.5,
            recurrent_dropout=0.5
        )
    )

    # Predict a tag for every token
    model.add(
        TimeDistributed(
            Dense(n_tags, activation="softmax")
        )
    )

    # Compile model
    model.compile(
        loss="categorical_crossentropy",
        optimizer="adam",
        metrics=["accuracy"]
    )

    model.summary()

    return model

In [16]:

def train_model(X, y, model):
    loss = []

    for i in range(2):
        print(f"Training epoch {i + 1}/2")

        hist = model.fit(
            X,
            y,
            batch_size=16,  # Reduced from 1000
            verbose=1,
            epochs=5,
            validation_split=0.2
        )

        loss.append(hist.history['loss'][0])

    return loss


In [17]:
results = pd.DataFrame()

# Create the model
model_bilstm_lstm = get_bilstm_lstm_model()

# Display architecture
plot_model(
    model_bilstm_lstm,
    show_shapes=True,
    show_layer_names=True
)

# Train model
results["with_add_lstm"] = train_model(
    train_tokens,
    np.array(train_tags),
    model_bilstm_lstm
)

Model: "sequential_3"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_3 (Embedding)         │ (None, 104, 64)        │     2,251,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_3 (Bidirectional) │ (None, 104, 128)       │        66,048 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm_7 (LSTM)                   │ (None, 104, 64)        │        49,408 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ time_distributed_3              │ (None, 104, 17)        │         1,105 │
│ (TimeDistributed)               │                        │               │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,368,017 (9.03 MB)

 Trainable params: 2,368,017 (9.03 MB)

 Non-trainable params: 0 (0.00 B)

You must install graphviz (see instructions at https://graphviz.gitlab.io/download/) for `plot_model` to work.
Training epoch 1/2
Epoch 1/5
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 257s 149ms/step - accuracy: 0.9749 - loss: 0.1039 - val_accuracy: 0.9861 - val_loss: 0.0475
Epoch 2/5
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 237s 147ms/step - accuracy: 0.9898 - loss: 0.0363 - val_accuracy: 0.9915 - val_loss: 0.0299
Epoch 3/5
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 236s 146ms/step - accuracy: 0.9929 - loss: 0.0249 - val_accuracy: 0.9924 - val_loss: 0.0260
Epoch 4/5
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 235s 145ms/step - accuracy: 0.9942 - loss: 0.0200 - val_accuracy: 0.9927 - val_loss: 0.0250
Epoch 5/5
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 236s 146ms/step - accuracy: 0.9949 - loss: 0.0171 - val_accuracy: 0.9927 - val_loss: 0.0251
Training epoch 2/2
Epoch 1/5
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 234s 144ms/step - accuracy: 0.9955 - loss: 0.0152 - val_accuracy: 0.9928 - val_loss: 0.0256
Epoch 2/5
1619/1619 ━━━━━━━━━━━━━━━━━━━━ 230s 142ms/s

In [3]:
import spacy
from spacy import displacy
nlp = spacy.load('en_core_web_sm')
text = nlp("""
This project was developed by Team 7.

They are students of 7th sem at Srinivas University
Institute of Engineering and Technology.

This project is for the Natural Language Processing subject..
""")
displacy.render(text, style = 'ent', jupyter=True)